In [1]:
import numpy as np
import pandas as pd
import copy
cp = lambda x: copy.deepcopy(x)

def make_ordinal(n):
    # Check if the number ends in 11, 12, or 13
    if 11 <= (n % 100) <= 13:
        suffix = "th"
    else:
        # Match the last digit to the correct suffix
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"

In [2]:
# country-capital data
df_name = "country"
df = pd.read_csv("data/countries.csv")[["name", "capital"]].dropna()
print('read:', len(df))
df.head()

read: 245


,name,capital
0,Afghanistan,Kabul
1,Aland Islands,Mariehamn
2,Albania,Tirana
3,Algeria,Algiers
4,American Samoa,Pago Pago


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
torch.set_grad_enabled(False)

from matplotlib import pyplot as plt
import seaborn as sns

from general_utils import (
  ModelAndTokenizer,
  make_inputs,
  decode_tokens,
  find_token_range,
  predict_from_input,
)

from patchscopes_utils import *
from tqdm import tqdm
tqdm.pandas()

# for path:
from accelerate import Accelerator
from accelerate.utils import set_seed
import argparse
import pickle 
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/chunma/miniforge3/envs/dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
parser = argparse.ArgumentParser()
parser.add_argument("--exp_name", type=str)
parser.add_argument("--output_dir", type=str, default="./experiments/")
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.1-8B")

# args = parser.parse_args(args=['--exp_name', 'llama3.1', '--model_name', 'meta-llama/Llama-3.1-8B'])
# args = parser.parse_args(args=['--exp_name', 'pythia70m', '--model_name', 'EleutherAI/pythia-70m'])
# args = parser.parse_args(args=['--exp_name', 'pythia410m', '--model_name', 'EleutherAI/pythia-410m'])
# args = parser.parse_args(args=['--exp_name', 'gemma3.270m', '--model_name', 'google/gemma-3-270m'])
# args = parser.parse_args(args=['--exp_name', 'gemma2.2b', '--model_name', 'google/gemma-2-2b'])
# args = parser.parse_args(args=['--exp_name', 'gemma.2b', '--model_name', 'google/gemma-2b'])
# args = parser.parse_args(args=['--exp_name', 'gemma2.9b', '--model_name', 'google/gemma-2-9b'])
args = parser.parse_args(args=['--exp_name', 'phi2', '--model_name', 'microsoft/phi-2'])
# args = parser.parse_args(args=['--exp_name', 'phi1', '--model_name', 'microsoft/phi-1'])
# args = parser.parse_args(args=['--exp_name', 'qwen2.5.1.5b', '--model_name', 'Qwen/Qwen2.5-1.5B'])

print(args)

set_seed(args.seed)
output_dir = os.path.join(args.output_dir, args.exp_name)
os.makedirs(output_dir, exist_ok=True)

Namespace(exp_name='phi2', output_dir='./experiments/', seed=42, model_name='microsoft/phi-2')


In [5]:
model_to_hook = {
    "EleutherAI/pythia-6.9b": set_hs_patch_hooks_neox,
    "EleutherAI/pythia-12b": set_hs_patch_hooks_neox,
    "meta-llama/Llama-2-13b-hf": set_hs_patch_hooks_llama,
    "lmsys/vicuna-7b-v1.5": set_hs_patch_hooks_llama,
    "./stable-vicuna-13b": set_hs_patch_hooks_llama,
    "CarperAI/stable-vicuna-13b-delta": set_hs_patch_hooks_llama,
    "EleutherAI/gpt-j-6b": set_hs_patch_hooks_gptj,
    
    "EleutherAI/pythia-70m": set_hs_patch_hooks_neox,
    "EleutherAI/pythia-410m": set_hs_patch_hooks_neox,    
    "google/gemma-2-2b": set_hs_patch_hooks_llama,
    "google/gemma-2b": set_hs_patch_hooks_llama,
    "google/gemma-3-270m": set_hs_patch_hooks_llama,
    "google/gemma-2-9b": set_hs_patch_hooks_llama,
    "meta-llama/Llama-3.1-8B": set_hs_patch_hooks_llama,
    "Qwen/Qwen2.5-1.5B": set_hs_patch_hooks_llama,    
    "microsoft/phi-2": set_hs_patch_hooks_phi,
    "microsoft/phi-1": set_hs_patch_hooks_phi
}

model_to_head = {
    "Eleu": "embed_out",
    "goog": "lm_head",
    "meta": "lm_head",
    "micr": "lm_head",
    "Qwen": "lm_head"
}

In [6]:
# Load model

model_name = args.model_name # "microsoft/phi-2" # "EleutherAI/pythia-70m" 
sos_tok = False

if "9b" in model_name or "8b" in model_name:
    torch_dtype = torch.bfloat16 # torch.float16
else:
    torch_dtype = None

my_device = torch.device("cuda:0") # mps cuda:1

mt = ModelAndTokenizer(
    model_name,
    low_cpu_mem_usage=False,
    torch_dtype=torch_dtype,
    device=my_device,
)
mt.set_hs_patch_hooks = model_to_hook[model_name]
mt.model.eval()

# NOTE: should check the name of the unembedding layer:

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 453/453 [00:00<00:00, 6346.77it/s]


PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((2560,), eps=1

In [7]:
# OPTIONAL! sanity check.
prompt = "The capital of France is"
inputs = mt.tokenizer(prompt,return_tensors="pt",).to(mt.model.device)
with torch.no_grad():
    outputs = mt.model(**inputs)

logits = outputs.logits[0, -1]

print("dtype:", logits.dtype)
print("has_nan:", torch.isnan(logits).any().item())
print("has_inf:", torch.isinf(logits).any().item())
print("max:", logits.float().max().item())
print("min:", logits.float().min().item())

probs = torch.softmax(logits.float(), dim=-1)

topk_probs, topk_ids = torch.topk(probs, 10)

for p, idx in zip(topk_probs, topk_ids):
    token = mt.tokenizer.convert_ids_to_tokens(int(idx))
    print(f"{token:20s} {p.item():.6f}")

dtype: torch.float16
has_nan: False
has_inf: False
max: 19.015625
min: -7.3984375
ĠParis               0.805154
"                    0.019384
:                    0.017788
ĠBerlin              0.014979
Ġ__                  0.010705
...                  0.009978
..."                 0.008081
ĠLondon              0.007711
Ġ_                   0.006544
Ġ                    0.006343


In [8]:
# ── Shared helpers ─────────────────────────────────────────────────────────────
t2n = lambda tensor: tensor.half().detach().cpu().numpy() # .half() for gemma-3-270m. Should we all use half??

def _resolve_correct_token_id(tokenizer, source_answer: str) -> int:
    """Encode source_answer and assert it is exactly one token."""
    ids = tokenizer.encode(source_answer, add_special_tokens=False)
    # adjust ids: if more than 1 tokens in the answer, use the 1st
    if len(ids) > 1:
        ids = ids[:1]
    assert len(ids) == 1, (
        f"source_answer {source_answer!r} encodes to {len(ids)} tokens; "
        "expected exactly 1."
    )
    return ids[0]

def _resolve_correct_token(tokenizer, source_answer: str) -> str:
    """Encode source_answer and return its first token as a string.
    
    see https://claude.ai/chat/4c08d33d-c237-489d-bff3-10dae51de4d5
    Also note a subtlety: tokenizer.encode and tokenizer.tokenize can diverge for some tokenizers (e.g., due to normalization, special-casing of the first token in a sequence, or stripping of leading spaces) even though they usually agree. Since this function mirrors _resolve_correct_token_id by using encode rather than tokenize, the returned token should stay consistent with whatever ID-based logic you had before — but if you plug this into the split_and_fill function from earlier (which used tokenize), you might see edge-case mismatches between the two. Want me to rewrite split_and_fill to use this same encode-based approach for full consistency?
    """
    ids = tokenizer.encode(source_answer, add_special_tokens=False)
    # adjust ids: if more than 1 tokens in the answer, use the 1st
    if len(ids) > 1:
        ids = ids[:1]
    assert len(ids) == 1, (
        f"source_answer {source_answer!r} encodes to {len(ids)} tokens; "
        "expected exactly 1."
    )
    return tokenizer.convert_ids_to_tokens(ids[0])

def _resolve_correct_id(tokenizer, source_answer: str) -> str:
    # here we already know that source_answer is a single token
    if len(source_answer) == 0:
        return tokenizer.unk_token_id # NOTE: if 'G' -> '', use unkonwn token (not a big deal to replace it)
    ID = tokenizer.convert_tokens_to_ids(source_answer)
    assert ID != tokenizer.unk_token_id
    return ID

def _compute_metrics(dist: torch.Tensor, correct_token_id: int, k: int, tokenizer):
    """
    Given a 1-D probability distribution over vocabulary, compute all metrics.

    Returns:
        topk_tokens       : list[str]  top-k token strings
        topk_probs        : list[float] top-k probabilities
        correct_token_rank: int        0-indexed rank of the correct token
        correct_token_prob: float      probability of the correct token
        nll               : float      negative log-likelihood of the correct token
    """
    # top-k
    topk_probs_t, topk_idx = torch.topk(dist, k)
    topk_tokens = tokenizer.convert_ids_to_tokens(topk_idx.tolist())
    topk_probs = topk_probs_t.detach().cpu().tolist()

    # correct-token metrics
    correct_token_prob = dist[correct_token_id].detach().cpu().item()
    correct_token_rank = (dist > dist[correct_token_id]).sum().detach().cpu().item()
    nll = -torch.log(dist[correct_token_id]).detach().cpu().item()

    return topk_tokens, topk_probs, correct_token_rank, correct_token_prob, nll

def _compute_metrics2(dist: torch.Tensor, correct_token_id: int, k: int, tokenizer):
    """
    copy of _compute_metrics, but only returns (correct_token_rank, correct_token_prob, nll)
    """    
    # correct-token metrics
    correct_token_prob = dist[correct_token_id].detach().cpu().item()
    correct_token_rank = (dist > dist[correct_token_id]).sum().detach().cpu().item()
    nll = -torch.log(dist[correct_token_id]).detach().cpu().item()

    return correct_token_rank, correct_token_prob, nll

def get_unembed_fn(model, hidden_only=False):
    """
    Build a callable  h -> logits  that applies:
        final_layer_norm -> lm_head -> soft_cap (if present)

    Covers: LLaMA/Mistral/Phi-3/Qwen/Gemma  (model.model.norm)
            GPT-2                             (model.transformer.ln_f)
            Pythia / GPT-NeoX                (model.gpt_neox.final_layer_norm)
            OPT                              (model.model.decoder.final_layer_norm)
    """
    final_norm = None
    for attr_path in [
        "model.norm",
        "transformer.ln_f",
        "gpt_neox.final_layer_norm",
        "model.decoder.final_layer_norm",
        "model.final_layernorm", # Phi-2
        "model.language_model.norm" # Gemma-3-12b
    ]:
        obj = model
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            final_norm = obj
            break
        except AttributeError:
            continue

    if final_norm is None:
        raise ValueError(
            "Could not find final layer norm for this model architecture. "
            "Please add the attribute path to get_unembed_fn()."
        )
    
    lm_head = getattr(model, model_to_head[model_name[:4]]) # not always lm_head
    cap = getattr(getattr(model, "config", None), "final_logit_softcapping", None)

    def unembed(h: torch.Tensor) -> torch.Tensor:
        """h: [d_model] — single position, single batch."""
        model_dtype = next(lm_head.parameters()).dtype
        h = h.to(model_dtype)
        h = final_norm(h)
        if hidden_only:
            return h
        logits = lm_head(h)
        if cap is not None:
            logits = torch.tanh(logits / cap) * cap
        return logits

    return unembed

# ── Function 1: standard greedy pass ──────────────────────────────────────────
def evaluate_regular(
    mt,
    prompt: str,
    source_answer: str,
    k: int = 10,
):
    """
    Single greedy forward pass (no sampling, no top-k/top-p, no temperature).
    Returns metrics at the final layer only.

    Returns:
        topk_tokens, topk_probs, correct_token_rank, correct_token_prob, nll
    """
    correct_token_id = _resolve_correct_token_id(mt.tokenizer, f" {source_answer}")
    correct_token_id2 = _resolve_correct_token_id(mt.tokenizer, source_answer)

    inp = make_inputs(mt.tokenizer, [prompt], mt.device)

    with torch.no_grad():
        output = mt.model(**inp)

    # greedy distribution at the last token position
    dist = torch.softmax(output.logits[0, -1, :], dim=0)

    return _compute_metrics(dist, correct_token_id, k, mt.tokenizer), _compute_metrics2(dist, correct_token_id2, k, mt.tokenizer)

# ── Function 2: Logit Lens ─────────────────────────────────────────────────────
def evaluate_logit_lens_topk(
    mt,
    prompt: str,
    source_answer: str,
    k: int = 10,
    return_hidden = False,
    monitor = [] # ['ĠFrance', 'Ġthe', 'Ġ..."', 'Ġin'] + ['France', 'the', '..."', 'in']
):
    """
    Logit Lens: apply  final_norm -> lm_head -> soft_cap  uniformly to every
    hidden state (including the last), at the last token position.

    Returns:
        List of per-layer tuples:
            (topk_tokens, topk_probs, correct_token_rank, correct_token_prob, nll)
        Length == mt.num_layers + 1  (embedding + one per transformer block)
    
    NOTE:
    h_layer[0, -1, :] on a tensor of shape (batch_size, seq_len, hidden_dim):
    0: batch index 0
    -1: last position in sequence
    :: all hidden dimensions
    """
    return_monitor = True if len(monitor) > 0 else False 
    if return_monitor:
        results_monitor = {t:[] for t in monitor}
        correct_token_monitor = {t:_resolve_correct_id(mt.tokenizer, t) for t in monitor} # NOTE: NOT _resolve_correct_token_id! which will do 'GParis' -> 'G' = 128!
    
    correct_token_id = _resolve_correct_token_id(mt.tokenizer, f" {source_answer}")
    correct_token_id2 = _resolve_correct_token_id(mt.tokenizer, source_answer)
    unembed = get_unembed_fn(mt.model)
    unembed_hidden = get_unembed_fn(mt.model, hidden_only=True)

    inp = make_inputs(mt.tokenizer, [prompt], mt.device)

    with torch.no_grad():
        output = mt.model(**inp, output_hidden_states=True)

    # hidden_states: tuple of (num_layers + 1) tensors, each [batch, seq, d_model]
    # index 0 = embedding output; index l = output of transformer block l-1
    results  = []
    results2 = []
    hiddens  = []
    for idh, h_layer in enumerate(output.hidden_states): # NOTE: the type for gemma-3-270m is "torch.bfloat16"
        h = h_layer[0, -1, :]          # [d_model] — last token, single batch
        if return_hidden:
            if idh < len(output.hidden_states) - 1:
                h_ = unembed_hidden(h) # note that last hidden will have DIFFERENT logic! norm twice!
            hiddens.append(t2n(h_))
        logits = unembed(h)            # [vocab_size]
        dist = torch.softmax(logits.to(torch.float32), dim=0)
        results.append(_compute_metrics(dist, correct_token_id, k, mt.tokenizer))
        results2.append(_compute_metrics2(dist, correct_token_id2, k, mt.tokenizer))
        if return_monitor:
            for t in monitor:
                results_monitor[t].append(_compute_metrics2(dist, correct_token_monitor[t], k, mt.tokenizer))
    
    if return_monitor:
        return results, results2, results_monitor, np.array(hiddens) # note that the 1st hidden is emb        
    if return_hidden: # always get hidden
        return results, results2, np.array(hiddens) # note that the 1st hidden is emb
    return results

# ── Function 3: Patchscopes ────────────────────────────────────────────────────
def evaluate_patch_next_token_prediction_topk(
    mt,
    prompt_source: str,
    prompt_target: str,
    layer_source: int,
    layer_target: int,
    position_source: int,
    position_target: int,
    source_answer: str,
    module: str = "hs",
    position_prediction: int = -1,
    transform=None,
    k: int = 10,
    return_hidden = False,
    monitor = [] # ['ĠFrance', 'Ġthe', 'Ġ..."', 'Ġin'] + ['France', 'the', '..."', 'in']    
):
    """
    Patchscopes: patch hidden state from layer_source of the source prompt
    into layer_target of the target prompt, then read off the distribution
    at position_prediction.

    Returns:
        topk_tokens, topk_probs, correct_token_rank, correct_token_prob, nll
    """
    return_monitor = True if len(monitor) > 0 else False 
    
    if module != "hs":
        raise ValueError("Module %s not yet supported" % module)

    correct_token_id = _resolve_correct_token_id(mt.tokenizer, f" {source_answer}")
    correct_token_id2 = _resolve_correct_token_id(mt.tokenizer, source_answer)    

    # ----- Prepare target input and resolve absolute position_target -----
    inp_target = make_inputs(mt.tokenizer, [prompt_target], mt.device)
    if position_target < 0:
        position_target = len(inp_target["input_ids"][0]) + position_target

    # ----- Source run: extract hidden rep -----
    inp_source = make_inputs(mt.tokenizer, [prompt_source], mt.device)
    with torch.no_grad():
        output_orig = mt.model(**inp_source, output_hidden_states=True)

    hidden_rep = output_orig["hidden_states"][layer_source + 1][0][position_source]
    if transform is not None:
        hidden_rep = transform(hidden_rep)

    # ----- Target run: patch and read distribution -----
    hs_patch_config = {layer_target: [(position_target, hidden_rep)]}
    skip_final_ln = (layer_source == layer_target == mt.num_layers - 1)

    patch_hooks = mt.set_hs_patch_hooks(
        mt.model,
        hs_patch_config,
        module=module,
        patch_input=False,
        skip_final_ln=skip_final_ln,
        generation_mode=True,
    )

    with torch.no_grad():
        output = mt.model(**inp_target, output_hidden_states=return_hidden)

    hidden_ps = output.hidden_states[-1][0, -1, :]
    hidden_ps = t2n(hidden_ps)

    dist = torch.softmax(output.logits[0, position_prediction, :], dim=0)
    remove_hooks(patch_hooks)

    if return_monitor:
        correct_token_monitor = {t:_resolve_correct_id(mt.tokenizer, t) for t in monitor}
        results_monitor = {t:_compute_metrics2(dist, correct_token_monitor[t], k, mt.tokenizer) for t in monitor}
        return _compute_metrics(dist, correct_token_id, k, mt.tokenizer), _compute_metrics2(dist, correct_token_id2, k, mt.tokenizer), results_monitor, hidden_ps        

    if return_hidden:
        return _compute_metrics(dist, correct_token_id, k, mt.tokenizer), _compute_metrics2(dist, correct_token_id2, k, mt.tokenizer), hidden_ps
    return _compute_metrics(dist, correct_token_id, k, mt.tokenizer)


In [9]:
def split_valid_pairs(tokens, tokenizer):
    vocab = tokenizer.get_vocab()

    marker_tokens = tokenizer.tokenize(" ")
    if not marker_tokens:
        raise ValueError("tokenizer.tokenize(' ') returned no tokens; can't infer the space marker.")
    if len(marker_tokens) > 1:
        raise ValueError(
            f"tokenizer.tokenize(' ') returned multiple tokens {marker_tokens}; "
            "this tokenizer may not use a simple prefix marker."
        )
    marker = marker_tokens[0]

    with_space, bare_list = [], []
    for tok in tokens:
        if tok.startswith(marker):
            withsp, bare = tok, tok[len(marker):]
        else:
            withsp, bare = marker + tok, tok

        if withsp in vocab and bare in vocab:
            with_space.append(withsp)
            bare_list.append(bare)

    return with_space, bare_list

def split_and_fill(tokens, tokenizer):
    vocab = tokenizer.get_vocab()

    marker_tokens = tokenizer.tokenize(" ")
    if not marker_tokens:
        raise ValueError("tokenizer.tokenize(' ') returned no tokens; can't infer the space marker.")
    if len(marker_tokens) > 1:
        raise ValueError(
            f"tokenizer.tokenize(' ') returned multiple tokens {marker_tokens}; "
            "this tokenizer may not use a simple prefix marker."
        )
    marker = marker_tokens[0]

    with_space_list, bare_list = [], []
    for tok in tokens:
        word = tok[len(marker):] if tok.startswith(marker) else tok

        with_space_str = marker + word
        bare_str = word

        if with_space_str not in vocab:
            sub = tokenizer.tokenize(" " + word)
            with_space_str = sub[0] if sub else with_space_str

        if bare_str not in vocab:
            sub = tokenizer.tokenize(word)
            bare_str = sub[0] if sub else bare_str

        # handle 'Ġ'
        if len(bare_str) == 0:
            bare_str = with_space_str
            
        with_space_list.append(with_space_str)
        bare_list.append(bare_str)

    return with_space_list, bare_list


In [10]:
from collections import defaultdict
rank_rg_, prob_rg_, nll_rg_, prob1_rg_, tok_rg_ = [], [], [], [], []
rank_ll_, prob_ll_, nll_ll_, prob1_ll_, tok_ll_ = [], [], [], [], []
rank_ps_, prob_ps_, nll_ps_, prob1_ps_, tok_ps_ = [], [], [], [], []
rank_rg_2, prob_rg_2, nll_rg_2 = [], [], []
rank_ll_2, prob_ll_2, nll_ll_2 = [], [], []
rank_ps_2, prob_ps_2, nll_ps_2 = [], [], []
prompts = []
answers = []
hiddens    = []
ps_hiddens = []
print_out = False # True # False 
k         = 5

tokens_monitor = []
result_monitor_ll = []
result_monitor_ps = []

correct_tokens = []

for irow in tqdm(range(80)): # (range(len(df))):
    row = df.iloc[irow]
    if df_name == 'country':
        # source_prompt = f"The capital of {row['name']} is"
        source_prompt = f"The capital of {row['name']} is" # NOTE: " is the city of " in the end
        source_answer = row['capital']
    elif df_name == 'chemistry':
        source_prompt = f"The element with the atomic number {row['atomic_number']} is"
        source_answer = row['answer']
    elif df_name == 'president':
        source_prompt = f"The {make_ordinal(row['S.No.'])} President of the United States was"
        source_answer = row['president']
    elif df_name in ['Paris', 'London', 'Berlin', 'Rome']:
        source_prompt = row['prompt']
        source_answer = row['answer']
    else:
        print('ERROR! check df_name.')
        break
    target_prompt = "cat -> cat\n1135 -> 1135\nhello -> hello\n?" # https://github.com/PAIR-code/interpretability/blob/master/patchscopes/code/next_token_prediction.ipynb

    prompts.append(source_prompt)
    answers.append(source_answer)
    
    correct_tokens.append(_resolve_correct_token(mt.tokenizer, f" {source_answer}"))
        
    # --- Regular pass (gold standard) ---
    (topk_tokens, topk_probs, rank, prob, nll), (rank2, prob2, nll2) = evaluate_regular(
        mt, source_prompt, source_answer, k=k
    )
    rank_rg_.append(rank)
    prob_rg_.append(prob)
    nll_rg_.append(nll)
    rank_rg_2.append(rank2)
    prob_rg_2.append(prob2)
    nll_rg_2.append(nll2)
    prob1_rg_.append(topk_probs[0])
    tok_rg_.append(topk_tokens[0])
    if print_out:
        print("=== Regular pass ===")
        print(f"rank: {rank}  prob: {prob:.4f}  nll: {nll:.4f}")
        for tok, p in zip(topk_tokens, topk_probs):
            print(f"  {tok:>15}  {p:.4f}")    

    rank_ll, prob_ll, nll_ll, prob1_ll, tok_ll = [], [], [], [], []
    rank_ll2, prob_ll2, nll_ll2 = [], [], []
    
    # get monitor tokens
    with_space, bare = split_and_fill(topk_tokens, mt.tokenizer)
    tokens_monitor.append((with_space, bare))
    
    # --- Logit Lens ---
    if print_out:
        print("\n=== Logit Lens ===")
    layer_results, layer_results2, results_monitor, hidden = evaluate_logit_lens_topk(mt, source_prompt, source_answer, k=k, 
                                                     return_hidden=True,
                                                     monitor=with_space+bare
                                                     )
    for layer, (topk_tokens, topk_probs, rank, prob, nll) in enumerate(layer_results):
        rank_ll.append(rank)
        prob_ll.append(prob)
        nll_ll.append(nll)
        prob1_ll.append(topk_probs[0])
        tok_ll.append(topk_tokens[0])
        if print_out:
            print(f"\n-- Layer {layer} --")
            print(f"rank: {rank}  prob: {prob:.4f}  nll: {nll:.4f}")
            for tok, p in zip(topk_tokens, topk_probs):
                print(f"  {tok:>15}  {p:.4f}")
    for layer, (rank2, prob2, nll2) in enumerate(layer_results2):
        rank_ll2.append(rank2)
        prob_ll2.append(prob2)
        nll_ll2.append(nll2)
    rank_ll_.append(rank_ll)
    prob_ll_.append(prob_ll)
    nll_ll_.append(nll_ll)
    rank_ll_2.append(rank_ll2)
    prob_ll_2.append(prob_ll2)
    nll_ll_2.append(nll_ll2)
    prob1_ll_.append(prob1_ll)
    tok_ll_.append(tok_ll)
    hiddens.append(hidden)
    result_monitor_ll.append(results_monitor)
    
    rank_ps, prob_ps, nll_ps, prob1_ps, tok_ps, hidden_ps = [], [], [], [], [], []
    rank_ps2, prob_ps2, nll_ps2 = [], [], []
    accum = defaultdict(list)
    
    # --- Patchscopes ---
    if print_out:
        print("\n=== Patchscopes ===")
    for layer in range(mt.num_layers):
        layer_results, layer_results2, results_monitor, hidden = evaluate_patch_next_token_prediction_topk(
            mt=mt,
            prompt_source=source_prompt,
            prompt_target=target_prompt,
            layer_source=layer,
            layer_target=layer,
            position_source=-1,
            position_target=-1,
            source_answer=source_answer,
            module="hs",
            position_prediction=-1,
            transform=None,
            k=k,
            return_hidden=True,
            monitor=with_space+bare            
        )
        topk_tokens, topk_probs, rank, prob, nll = layer_results
        rank_ps.append(rank)
        prob_ps.append(prob)
        nll_ps.append(nll)
        prob1_ps.append(topk_probs[0])
        tok_ps.append(topk_tokens[0])
        hidden_ps.append(hidden)
        rank2, prob2, nll2 = layer_results2
        rank_ps2.append(rank2)
        prob_ps2.append(prob2)
        nll_ps2.append(nll2)
        if print_out:
            print(f"\n-- Layer {layer} --")
            print(f"rank: {rank}  prob: {prob:.4f}  nll: {nll:.4f}")
            for tok, p in zip(topk_tokens, topk_probs):
                print(f"  {tok:>15}  {p:.4f}")
        for kay, vala in results_monitor.items():
            accum[kay].append(vala)
    rank_ps_.append(rank_ps)
    prob_ps_.append(prob_ps)
    nll_ps_.append(nll_ps)
    rank_ps_2.append(rank_ps2)
    prob_ps_2.append(prob_ps2)
    nll_ps_2.append(nll_ps2)
    prob1_ps_.append(prob1_ps)
    tok_ps_.append(tok_ps)
    ps_hiddens.append(np.array(hidden_ps))
    result_monitor_ps.append(accum)
    
    # break
# L40: 9 min for 245 items

100%|██████████| 80/80 [02:54<00:00,  2.19s/it]


In [11]:
print(tokens_monitor[0])
print(correct_tokens[0])
print(len(result_monitor_ll))
print(len(result_monitor_ps))

(['ĠKabul', 'Ġlocated', 'Ġthe', 'Ġ..."', 'Ġin'], ['K', 'l', 'the', '..."', 'in'])
ĠKabul
80
80


In [12]:
idx = 0
print(hiddens[idx].shape)
print(ps_hiddens[idx].shape)
print(prompts[idx], '->', answers[idx])

(33, 2560)
(32, 2560)
The capital of Afghanistan is -> Kabul


In [13]:
print(np.mean(np.array(rank_rg_) == 0))
print(np.mean(np.array(rank_rg_) <= 1))
print(np.mean(np.array(rank_rg_) <= 3))
print(np.mean(np.array(rank_rg_) <= 5))
print(np.mean(np.array(rank_rg_) <= 20))

0.8875
0.9375
0.95
0.9625
0.975


In [14]:
dfile = os.path.join(output_dir, f"lens_{df_name}.pkl")
with open(dfile, "wb") as file:
    pickle.dump(prompts, file)
    pickle.dump(answers, file)    
    pickle.dump(hiddens, file)
    pickle.dump(ps_hiddens, file)
    pickle.dump((rank_rg_, prob_rg_, nll_rg_, prob1_rg_, tok_rg_, rank_rg_2, prob_rg_2, nll_rg_2), file)
    pickle.dump((rank_ll_, prob_ll_, nll_ll_, prob1_ll_, tok_ll_, rank_ll_2, prob_ll_2, nll_ll_2), file)
    pickle.dump((rank_ps_, prob_ps_, nll_ps_, prob1_ps_, tok_ps_, rank_ps_2, prob_ps_2, nll_ps_2), file)
    
    pickle.dump(tokens_monitor, file)    
    pickle.dump(result_monitor_ll, file)    
    pickle.dump(result_monitor_ps, file)    
    
    pickle.dump(correct_tokens, file)
    
print('saved to:', dfile, len(prompts))

saved to: ./experiments/phi2/lens_country.pkl 80
